In [4]:
import duckdb
import json
from pathlib import Path
import networkx as nx
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModel
import faiss

print("Imports OK")


Imports OK


In [5]:
# Path to one Spider DB (academic for now)
db_path = Path("../data/spider/database/academic/academic.sqlite").resolve()

if not db_path.exists():
    raise FileNotFoundError(db_path)

con = duckdb.connect(database=":memory:")

con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

con.execute(f"""
ATTACH DATABASE '{db_path}' AS spider_db (TYPE sqlite);
""")

print("Spider DB attached")


Spider DB attached


In [6]:
# Get tables
tables = con.execute("""
SELECT table_name
FROM duckdb_tables()
WHERE database_name = 'spider_db';
""").fetchall()

table_names = [t[0] for t in tables]
print("Tables found:", table_names)


Tables found: ['author', 'conference', 'domain', 'domain_author', 'domain_conference', 'journal', 'domain_journal', 'keyword', 'domain_keyword', 'publication', 'domain_publication', 'organization', 'publication_keyword', 'writes', 'cite']


In [7]:
schema = {}

for table in table_names:
    cols = con.execute(f"""
        SELECT column_name, data_type
        FROM duckdb_columns()
        WHERE database_name = 'spider_db'
          AND table_name = '{table}';
    """).fetchall()

    schema[table] = [
        {"column": c[0], "type": c[1]}
        for c in cols
    ]

print("Schema extracted for", len(schema), "tables")


Schema extracted for 15 tables


In [8]:
G = nx.Graph()


for table in schema:
    G.add_node(table, type="table")

for table, cols in schema.items():
    for col in cols:
        col_node = f"{table}.{col['column']}"
        G.add_node(col_node, type="column")
        G.add_edge(table, col_node, relation="has_column")

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())


Graph nodes: 57
Graph edges: 42


In [9]:
fk_info = []

for table in table_names:
    try:
        fks = con.execute(f"PRAGMA spider_db.foreign_key_list('{table}');").fetchall()
        for fk in fks:
            fk_info.append({
                "from_table": table,
                "from_col": fk[3],
                "to_table": fk[2],
                "to_col": fk[4]
            })
    except:
        pass

print("Foreign keys found:", len(fk_info))


Foreign keys found: 0


In [10]:
for fk in fk_info:
    src = f"{fk['from_table']}.{fk['from_col']}"
    tgt = f"{fk['to_table']}.{fk['to_col']}"

    if src in G and tgt in G:
        G.add_edge(src, tgt, relation="FK")

print("Graph updated with FK edges")


Graph updated with FK edges


In [11]:
texts = []
ids = []

for table, cols in schema.items():
    table_text = f"table {table} with columns " + ", ".join(
        c["column"] for c in cols
    )
    texts.append(table_text)
    ids.append(table)

    for c in cols:
        col_text = f"column {c['column']} in table {table} of type {c['type']}"
        texts.append(col_text)
        ids.append(f"{table}.{c['column']}")

print("Embedding items:", len(texts))


Embedding items: 57


In [14]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

class HFTextEmbedder:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

    @torch.no_grad()
    def encode(self, texts, normalize_embeddings: bool = True):
        if isinstance(texts, str):
            texts = [texts]

        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        outputs = self.model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
        embeddings = embeddings.cpu().numpy()

        if normalize_embeddings:
            embeddings = embeddings / np.linalg.norm(
                embeddings, axis=1, keepdims=True
            )

        return embeddings


In [15]:
embedder = HFTextEmbedder(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True
).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 57


In [16]:
def retrieve_schema(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(q_emb, k)

    return [(ids[i], float(scores[0][j])) for j, i in enumerate(idxs[0])]

# Test
query = "students enrolled in courses"
results = retrieve_schema(query)

print("Query:", query)
for r in results:
    print(r)


Query: students enrolled in courses
('domain_keyword', 0.134489506483078)
('domain_conference', 0.13314439356327057)
('organization.homepage', 0.13253669440746307)
('conference', 0.13222241401672363)
('domain_author', 0.1285504400730133)


In [17]:
import pickle

graph_path = Path("../schemas/schema_graph_academic.pkl")

with open(graph_path, "wb") as f:
    pickle.dump(G, f)

print("Graph saved to", graph_path)


Graph saved to ..\schemas\schema_graph_academic.pkl
